This notebook will conduct the preprocessing of the text data collected for the Kitwe project. The motivation is to uset this text data to classify the document as fake and genuine. In order to have a better understanding of the dataset we will try the following methods:

### Data Cleaning
1. Removing HTML tags
2. Tokenization
4. Converting date and time to a standard format
5. lemmatization
7. removing duplicate entries
8. removing punctuations (.,:-_()[]?''""!)

### Data Annotation
This involves classifying the data into genuine and fake based on various indicators derived from the raw text data

### Categorization of the data
The categories of the collected data is really ambiguous and needs to categorized well to understand the distribution.

For this project, we collected text data different News sources in Zambia using RSS feeds. Now we will clean, preprocess, annotate and categorize the data.

In [1]:
#import regex
import spacy
import re
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup as bf
from textblob import TextBlob
from urllib.parse import urlparse
from matplotlib import pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import KNeighborsClassifier

In [2]:
# Read the raw text data for clearning
df1 = pd.read_csv('../../data/raw-new.csv')
df2 = pd.read_csv('../../data/raw_old.csv')

In [3]:
df1.columns

Index(['Source', 'Category', 'Headline', 'Link', 'Description', 'Date',
       'Author'],
      dtype='object')

In [4]:
df2.columns

Index(['Source', 'Category', 'Headline', 'Link', 'Description', 'Date',
       'Author'],
      dtype='object')

In [5]:
df = pd.concat((df1, df2), axis=0)
df.shape

(16570, 7)

For text classification, the important columns are the headline and description. The source of the data and author information are important for classifying if the news is fake or not. For the moment, we can defnitely get rid of the link column. We will also convert the date to standard pandas date format so that we can get an idea of when the news was published.

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 16570 entries, 0 to 14343
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Source       16570 non-null  object
 1   Category     16096 non-null  object
 2   Headline     16569 non-null  object
 3   Link         16570 non-null  object
 4   Description  16550 non-null  object
 5   Date         16570 non-null  object
 6   Author       16569 non-null  object
dtypes: object(7)
memory usage: 1.0+ MB


In [7]:
df.columns

Index(['Source', 'Category', 'Headline', 'Link', 'Description', 'Date',
       'Author'],
      dtype='object')

In [8]:
# converting the date columnn to pandas date and time format
df['Date'] = pd.to_datetime(df['Date'])
df['Date'].iloc[:5]

0   2024-11-20 16:31:18+00:00
1   2024-11-20 15:15:28+00:00
2   2024-11-20 15:14:39+00:00
3   2024-11-20 15:13:41+00:00
4   2024-11-20 15:13:05+00:00
Name: Date, dtype: datetime64[ns, UTC]

In [9]:
# let's check for Nan values and duplicate entries
df.isna().sum()

Source           0
Category       474
Headline         1
Link             0
Description     20
Date             0
Author           1
dtype: int64

In [10]:
df[df['Category'].isna()]

,Source,Category,Headline,Link,Description,Date,Author
2013,Flava FM,NaN,Contact Us,https://flavaradioandtv.com/contact-us?utm_sou...,<p>Get In Touch Location: 3rd Floor Kitwe Main...,2021-01-19 08:43:25+00:00,NaN
2179,Copperbelt Energy,NaN,Businesses,https://cecinvestor.com/businesses/,Business segments Local Power Supply Source a...,2017-01-17 06:54:37+00:00,aiciadmin
2180,Copperbelt Energy,NaN,Management,https://cecinvestor.com/how-we-are-governed/ma...,How We Are Governed Management Most of CEC’s...,2017-01-15 06:01:43+00:00,aiciadmin
2181,Copperbelt Energy,NaN,Contact,https://cecinvestor.com/contact/,Investor Relations Precious M. Chisenga Corpor...,2017-01-12 06:10:53+00:00,aiciadmin
8546,Kitwe Online,NaN,TONGA LANGUAGE,https://kitweonline.com/languages/tonga-langua...,Resources: Here are some of the resources we h...,2022-10-20 10:57:00+00:00,JS
...,...,...,...,...,...,...,...
10670,Daily Nations Zambia,NaN,NaN,https://dailynationzambia.com/2021/03/10304/?u...,"Mon, 30 Nov -0001 00:00:00 +0000 WARRIORS HOLD...",2021-03-07 10:31:11+00:00,Daily Nation
10671,Daily Nations Zambia,NaN,FANS WANT POWER DYNAMOS HEAD COACH DISMISSED,https://dailynationzambia.com/2021/03/fans-wan...,"Mon, 30 Nov -0001 00:00:00 +0000 FANS WANT POW...",2021-03-07 10:31:11+00:00,Daily Nation
10672,Daily Nations Zambia,NaN,YOU CAN BUY THIS BOOK- A PRESIDENT BETRAYED FR...,https://dailynationzambia.com/2021/03/you-can-...,"Mon, 30 Nov -0001 00:00:00 +0000 Author: Richa...",2021-03-07 10:30:56+00:00,Daily Nation
10673,Daily Nations Zambia,NaN,"Phoenix gives KCC K32, 000 to help keep Kitwe ...",https://dailynationzambia.com/2021/03/phoenix-...,"Sat, 04 Feb 2017 11:29:22 +0000 By ROGERS KALE...",2021-03-07 10:30:56+00:00,Daily Nation


Okay. So there are plenty of null values in the category column and also the description ones.

In [11]:
# Look for duplicated entries
df.duplicated()

0        False
1        False
2        False
3        False
4        False
         ...  
14339    False
14340    False
14341    False
14342    False
14343    False
Length: 16570, dtype: bool

In [12]:
# Let's drop the duplicates
df.drop_duplicates(inplace=True)
df.shape

(14282, 7)

In [13]:
# Fill out NaN values
df.fillna(value = "", inplace=True)

In [14]:
df.isna().sum()

Source         0
Category       0
Headline       0
Link           0
Description    0
Date           0
Author         0
dtype: int64

In [15]:
# Let's take a look at the unique sources, caegories of the news
df['Source'].value_counts()

Source
Lusaka Times               6479
Lusaka Voice               1745
Daily Nations Zambia       1361
Zambia Eye                 1022
Kitwe Online                889
Mwebantu                    711
Zambia Monitor              627
Copperbelt Energy           445
Zambian Eye                 300
Daily Revelation Zambia     202
Zambia365                   130
Zambia Reports              119
News Invasion 24             84
Lusaka Star                  67
ZNBC                         64
Flava FM                     14
DailyMail                    10
Tech Africa News              7
Christian Voice               4
Daily Mail Zambia             2
Name: count, dtype: int64

In [16]:
# Look at the unique categories
df['Category'].unique()[:50]

array(['Careers,Current Careers',
       'Corporate Announcements,Downloads,Featured,Green Bond', 'Careers',
       'Corporate Social Responsibility,Featured',
       'Corporate Announcements,Featured', 'Featured,News',
       'Annual Report,Corporate Announcements,Downloads,Featured,AF',
       'Corporate Announcements,Featured,AF',
       'Corporate Announcements,Downloads,Featured,AF', 'News',
       'Corporate Social Responsibility,Featured,News,AF',
       'Corporate Social Responsibility,News',
       'Corporate Social Responsibility,Featured,News,Power Dynamos',
       'Corporate Social Responsibility,Power Dynamos',
       'Downloads,Featured,News', 'Careers,Featured,Power Dynamos',
       'Corporate Social Responsibility,News,Power Dynamos',
       'Corporate Announcements,Downloads,Featured', 'Power Dynamos',
       'Corporate Social Responsibility,Featured,News',
       'Annual Report,Corporate Announcements,Downloads,Featured',
       'Featured,News,Project', 'Corporate Ann

It looks like the Category column consists of many entries, ambiguous and often mixed with headline column for many of the training data. Therefore it is important to come up with a unique set of categories and appending that information to the Category column. We will do the categorization of the data towards the end.

### Text Preprocessing with Spacy

In [17]:
# Loading the spacy english language model
# This model was trained on large corpus of labeled text data. Labels like sentence parsing, POS tagging, named entity recongnition 
nlp = spacy.load('en_core_web_sm')

In [18]:
# Try out spacy in a single text before the whole training examples
# Now we will go through the description of the training example for preprocessing
s = df['Description'].iloc[662]
s

'Nkana on Saturday made a massive leap in their battle to survive relegation when they rallied to beat old foes Mighty Mufulira Wanderers 2-1 at home in Kitwe. The victory in Wusakile saw Nkana rise from the relegation trap door at number 15 to 11 on 29 points, one point behind Mighty who lost for [&#8230;]'

In [19]:
doc = nlp(s)

# Getting tokens and their indexes
tok = [(t.i, t.text) for t in doc]
tok

[(0, 'Nkana'),
 (1, 'on'),
 (2, 'Saturday'),
 (3, 'made'),
 (4, 'a'),
 (5, 'massive'),
 (6, 'leap'),
 (7, 'in'),
 (8, 'their'),
 (9, 'battle'),
 (10, 'to'),
 (11, 'survive'),
 (12, 'relegation'),
 (13, 'when'),
 (14, 'they'),
 (15, 'rallied'),
 (16, 'to'),
 (17, 'beat'),
 (18, 'old'),
 (19, 'foes'),
 (20, 'Mighty'),
 (21, 'Mufulira'),
 (22, 'Wanderers'),
 (23, '2'),
 (24, '-'),
 (25, '1'),
 (26, 'at'),
 (27, 'home'),
 (28, 'in'),
 (29, 'Kitwe'),
 (30, '.'),
 (31, 'The'),
 (32, 'victory'),
 (33, 'in'),
 (34, 'Wusakile'),
 (35, 'saw'),
 (36, 'Nkana'),
 (37, 'rise'),
 (38, 'from'),
 (39, 'the'),
 (40, 'relegation'),
 (41, 'trap'),
 (42, 'door'),
 (43, 'at'),
 (44, 'number'),
 (45, '15'),
 (46, 'to'),
 (47, '11'),
 (48, 'on'),
 (49, '29'),
 (50, 'points'),
 (51, ','),
 (52, 'one'),
 (53, 'point'),
 (54, 'behind'),
 (55, 'Mighty'),
 (56, 'who'),
 (57, 'lost'),
 (58, 'for'),
 (59, '['),
 (60, '&'),
 (61, '#'),
 (62, '8230'),
 (63, ';'),
 (64, ']')]

In [20]:
# Getting each sentences as well
print([sent for sent in doc.sents])

[Nkana on Saturday made a massive leap in their battle to survive relegation when they rallied to beat old foes Mighty Mufulira Wanderers 2-1 at home in Kitwe., The victory in Wusakile saw Nkana rise from the relegation trap door at number 15 to 11 on 29 points, one point behind Mighty who lost for [&#8230;]]


In [21]:
# case folding
print([t.lower_ for t in doc])

['nkana', 'on', 'saturday', 'made', 'a', 'massive', 'leap', 'in', 'their', 'battle', 'to', 'survive', 'relegation', 'when', 'they', 'rallied', 'to', 'beat', 'old', 'foes', 'mighty', 'mufulira', 'wanderers', '2', '-', '1', 'at', 'home', 'in', 'kitwe', '.', 'the', 'victory', 'in', 'wusakile', 'saw', 'nkana', 'rise', 'from', 'the', 'relegation', 'trap', 'door', 'at', 'number', '15', 'to', '11', 'on', '29', 'points', ',', 'one', 'point', 'behind', 'mighty', 'who', 'lost', 'for', '[', '&', '#', '8230', ';', ']']


In [22]:
# stop word removal
# Let's look at the spacy's default list of stop word list
print(nlp.Defaults.stop_words)

{'regarding', 'have', 'keep', 'besides', 'them', 'thereafter', 'thereby', 'last', 'often', 'at', 'becoming', 'beside', '’d', 'next', 'also', '‘d', 'anywhere', 'none', 'it', 'upon', "'m", 'therefore', 'then', 'nevertheless', 'fifteen', 'third', 'yourself', 'a', 'under', 'those', 'why', 'already', 'becomes', 'however', 'me', 'nine', 'elsewhere', 'what', '’ll', 'whole', 'although', 'amongst', "n't", 'due', 'which', 'the', 'sixty', 'sometimes', 'while', 'too', 'on', '‘ve', 'were', 'been', 'but', 'enough', 'nor', 'most', 'no', 'well', 'themselves', 'namely', 'my', '’ve', 'either', 'made', 'put', 'hereby', 'as', 'nowhere', 'up', 'we', 'throughout', 'being', 'if', 'here', 'every', 'among', '‘re', 'and', 'really', 'whereby', 'whom', 'is', 'everywhere', 'whoever', 'must', 'five', 'very', 'full', 'had', 'mine', "'re", 'further', 'himself', 'twenty', 'indeed', 'nothing', '’s', 'bottom', 'even', 'alone', 'ever', 'above', 'through', 'ca', 'latterly', 'i', 'how', '‘ll', 'please', 'for', 'move', 'som

In [23]:
print(len(nlp.Defaults.stop_words))

326


Interstingly, there are some words in the list which I won't consider to be stopping words.

In [24]:
# Remove the common occuring stop words here
# Remove punctuations as well
# NOw look at tokens which are not in the list
print([t.text for t in doc if (not t.is_stop) and (not t.is_punct)])

['Nkana', 'Saturday', 'massive', 'leap', 'battle', 'survive', 'relegation', 'rallied', 'beat', 'old', 'foes', 'Mighty', 'Mufulira', 'Wanderers', '2', '1', 'home', 'Kitwe', 'victory', 'Wusakile', 'saw', 'Nkana', 'rise', 'relegation', 'trap', 'door', 'number', '15', '11', '29', 'points', 'point', 'Mighty', 'lost', '8230']


### Cleaning of the data

In [25]:
# Function to remove html tags
def strip_html_tags(text):
    """
    Strip html tags in a text
    """
    soup = bf(text, "html.parser")
    stripped_text = soup.get_text()
    return stripped_text

In [26]:
def preprocess_text(text):
    """
    Tokenization, lemmatization, removal of stop words,
    punctuations and spaces
    """
    doc = nlp(text)
    
    # for lower case
    #return [tok.lemma_.lower() for tok in doc if (tok.is_alpha) and (not tok.is_stop) and (not tok.is_punct) and (not tok.is_space)]
    
    return [tok.lemma_ for tok in doc if (tok.is_alpha) and (not tok.is_stop) and (not tok.is_punct) and (not tok.is_space)]

In [27]:
print(preprocess_text(s))

['Nkana', 'Saturday', 'massive', 'leap', 'battle', 'survive', 'relegation', 'rally', 'beat', 'old', 'foe', 'Mighty', 'Mufulira', 'Wanderers', 'home', 'Kitwe', 'victory', 'Wusakile', 'see', 'Nkana', 'rise', 'relegation', 'trap', 'door', 'number', 'point', 'point', 'Mighty', 'lose']


In [28]:
# Let's make a copy of the dataset before doing the preprocessing
df_clean = df.copy()

# Now let's apply this function on the headline and the description column
df_clean['Headline'] = df_clean['Headline'].astype('str').apply(strip_html_tags)
df_clean['Headline'] = df_clean['Headline'].apply(preprocess_text)

df_clean['Description'] = df_clean['Description'].astype('str').apply(strip_html_tags)
df_clean['Description'] = df_clean['Description'].apply(preprocess_text)

/tmp/ipykernel_108557/1132794250.py:6: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  soup = bf(text, "html.parser")
/tmp/ipykernel_108557/1132794250.py:6: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  soup = bf(text, "html.parser")


In [29]:
df_clean.head(5)

,Source,Category,Headline,Link,Description,Date,Author
0,Copperbelt Energy,"Careers,Current Careers","[CEC, Career, Opportunity, GDP, Legal]",https://cecinvestor.com/search/kitwe/feed/rss2/,"[currently, career, opportunity, follow, field...",2024-11-20 16:31:18+00:00,Lovejoy Musundire
1,Copperbelt Energy,"Careers,Current Careers","[CEC, Career, Opportunity, Assurance, Speciali...",https://cecinvestor.com/search/kitwe/feed/rss2/,"[currently, career, opportunity, follow, field...",2024-11-20 15:15:28+00:00,Lovejoy Musundire
2,Copperbelt Energy,"Careers,Current Careers","[CEC, Career, Opportunity, Internal, Auditor, ...",https://cecinvestor.com/search/kitwe/feed/rss2/,"[currently, career, opportunity, follow, field...",2024-11-20 15:14:39+00:00,Lovejoy Musundire
3,Copperbelt Energy,"Careers,Current Careers","[CEC, Career, Opportunity, Engineer, protection]",https://cecinvestor.com/search/kitwe/feed/rss2/,"[currently, career, opportunity, follow, field...",2024-11-20 15:13:41+00:00,Lovejoy Musundire
4,Copperbelt Energy,"Careers,Current Careers","[CEC, Career, Opportunity, Mechanic, II]",https://cecinvestor.com/search/kitwe/feed/rss2/,"[currently, career, opportunity, follow, field...",2024-11-20 15:13:05+00:00,Lovejoy Musundire


### Label annotation
Now since we have finished the data cleaning process, the next important thing is to annotate the data as real or fake. We need to come up with a clear strategy of how to do that.
1. Checking if the source is a reliable and reputable news channel
2. Check if the news domain is suspicious
3. Look for clickbaits which has exaggerated use of sensational keywords
4. Check if the headline and description matches each other
5. check the polarity of the headline or the description, if its too negative
6. Check for execessive capitalization
7. Check for vague authors
8. Check for suspicious links

Let's classify the news as fake if it satisfies atleast 2 of these criterias.

In [30]:
class FakeNewsDetector():
    """
    Class to detect the genuinity of news elements
    within a given dataframe
    """

    def __init__(self, df):
        self.df = df # pandas data frame
        self.zambian_reputable_sources = ['daily-mail.co.zm', 'times.co.zm', 'znbc.co.zm', 'flavaradioandtv.com', 
            'lusakatimes.com', 'kitwetimes.com','zambiamonitor.com']
        self.suspicious_domain_pattern = re.compile(r'\\.(info|lo|ru|cn|xyz|top|news|live|buzz|click|online)$')
        
        # list of sensational words
        self.sensational_keywords = [
            'shocking', 'unbelievable', 'amazing', 'incredible', 'secret', 
            'exposed', 'you won’t believe', 'scandal', 'controversy'
        ]
    def check_vague_author(self, author):
        """
        Checking for vague authors
        """
        vague_authors = ['admin', 'editor', 'newsroom', 'staff', 'unknown']
        return 1 if any([vague_author in author.lower() for vague_author in vague_authors]) else 0
            
            
    def similarity_head_desc(self, row):
        """
        Try to find a cosine similarity between headline and
        description using tf-idf
        """
        comb_col = [row['Headline'], row['Description']]
        comb_txt = [" ".join(item) for item in comb_col] # This step is needed as we are feeding tokens
        # tfidf needs text without tokenizing as it will do the tokenizing, create the unique vocabulary and the vectorization
        
        tfidf_vectorizer = TfidfVectorizer()
        # This step will basically create a unique vocabulary and a matrix with features for each corpuse
        # elements. Or basically create a tf-idf vector for each row based on how frequency a token appears in one document and how the same 
        # appear in other documents
        tfidf_mat = tfidf_vectorizer.fit_transform(comb_txt)
        sim_score = cosine_similarity(tfidf_mat[0,:], tfidf_mat[1,:])

        return 1 if sim_score[0][0] < 0.10 else 0

    def check_source_credibility(self, url):
        """
        Check to see if the the netlocation of the URL
        is legit. For this parse the url and get the netloc infor. Then compare
        with predefined list
        """
        parsed_url = urlparse(url)
        domain = parsed_url.netloc.lower()
        return 1 if domain not in self.zambian_reputable_sources else 0
            
        
        
    def detect_clickbait(self, headline):
        """
        Look for excessive punctuations, all capital headlines
        Look for excessive usage of provocative words
        """
        headline = " ".join(headline)
        excessive_punctuation = len(re.findall(r'[!?.]{2,}', headline)) > 0
        all_caps = headline.isupper()
        provocative_words = any(word in headline.lower() for word in [
            'shocking', 'unbelievable', 'you won’t believe', 'secret', 
            'amazing', 'incredible'
        ])
        return 1 if excessive_punctuation or all_caps or provocative_words else 0
    

    def count_sensational_keywords(self, description):
        """
        Count the number of sensational words in the description to make sure
        it's not filled with them
        """
        desciption = " ".join(description)
        return sum(description.lower().count(word) for word in self.sensational_keywords)

    
    def get_sentiment_score(self, text):
        """
        Get the sentiment of the text if its positive or
        negative. Score goes from -1 to +1
        """
        text = " ".join(text)
        try:
            sentiment = TextBlob(text).sentiment
            return 1 if abs(sentiment.polarity) > 0.5 else 0
        except Exception as e:
            return 0
            
    def check_excessive_capitalization(self, text):
        """
        Check for excessive capitalization in the 
        headline or description. Could be indication of fake news
        """
        #words = text.split()
        capitalized_words = [word for word in text if word.isupper() and len(word) > 1]
        return 1 if len(capitalized_words) > 3 else 0 

    def count_suspicious_links(self, description):
        """
        Count the number of suspicious links in the description
        """
        desciption = " ".join(description)
        urls = re.findall(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\\\(\\\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', description)
        return 1 if len(urls) > 0 else 0
        
    def count_sensational_keywords(self, description):
        """
        Count how many keywords are in the description
        """
        
        return sum(description.lower().count(word) for word in self.sensational_keywords)
        
    def check_short_sensational_description(self, description):
        """
        Check for short sensational description
        """
        description_length = len(description)
        sensational_word_count = self.count_sensational_keywords(description)
        return 1 if description_length < 100 and sensational_word_count > 1 else 0

    def collect_fake_columns(self):
        
        # Fake author column
        self.df['fake_author'] = self.df['Author'].astype('str').apply(self.check_vague_author)

        # Similarity between source and descriptions
        self.df['dissimilar_head_desc'] = self.df.apply(self.similarity_head_desc, axis=1)

        # Legit URL netlocation
        self.df['fake_url'] = self.df['Link'].apply(self.check_source_credibility)

        # check for clickbaits
        self.df['click_bait'] = self.df['Headline'].astype('str').apply(self.detect_clickbait)
        #print(any(self.df['Headline'].apply(type) != 'str'))

        # sentiment score
        self.df['polar'] = self.df['Description'].astype('str').apply(self.get_sentiment_score)

        # excessive capitalization in the headline
        self.df['excess_capitalization'] = self.df['Headline'].astype('str').apply(self.check_excessive_capitalization)

        #Check for suspicious links
        self.df['susp_links'] = self.df['Description'].astype('str').apply(self.count_suspicious_links)

        # Check for short sensational description
        self.df['sens_description'] = self.df['Description'].astype('str').apply(self.check_short_sensational_description)
        
    def get_final_label(self):

        # replacing boolean with binary values
        #self.df.replace({True:1, False:0}, inplace=True)
        self.collect_fake_columns() # run the collection first
        self.df['fake_indicators'] = self.df.iloc[:,7:14].apply(np.sum, axis=1).astype('int')
        self.df['Target'] = self.df['fake_indicators'].copy()
        self.df['Target'] = self.df['Target'].apply(lambda x: 1 if x >=2 else 0)

In [31]:
ob = FakeNewsDetector(df_clean)
ob.get_final_label()

In [32]:
df_clean.iloc[200:222,:]

,Source,Category,Headline,Link,Description,Date,Author,fake_author,dissimilar_head_desc,fake_url,click_bait,polar,excess_capitalization,susp_links,sens_description,fake_indicators,Target
200,Copperbelt Energy,Corporate Announcements,"[CEC, Notice, Agenda, Seventeenth, Annual, Gen...",https://cecinvestor.com/search/kitwe/feed/rss2/,"[notice, give, Seventeenth, Annual, General, M...",2015-03-06 14:55:56+00:00,aiciadmin,1,0,1,0,0,0,0,0,2,1
201,Copperbelt Energy,Tenders,"[CEC, Invitation, tender, CEC, cost, service, ...",https://cecinvestor.com/search/kitwe/feed/rss2/,"[Copperbelt, Energy, Corporation, PLC, CEC, in...",2015-02-02 09:09:58+00:00,aiciadmin,1,0,1,0,0,0,0,0,2,1
202,Copperbelt Energy,Corporate Social Responsibility,"[CEC, Renewable, Energy, Essay, Writing, Compe...",https://cecinvestor.com/search/kitwe/feed/rss2/,"[Copperbelt, Energy, Corporation, Plc, CEC, pl...",2015-01-15 13:55:19+00:00,aiciadmin,1,0,1,0,0,0,0,0,2,1
203,Copperbelt Energy,Corporate Social Responsibility,"[CEC, Renewable, Energy, Essay, Writing, compe...",https://cecinvestor.com/search/kitwe/feed/rss2/,"[Copperbelt, Energy, Corporation, Plc, CEC, in...",2014-10-23 09:06:21+00:00,aiciadmin,1,0,1,0,0,0,0,0,2,1
204,Copperbelt Energy,Careers,"[vacancy, Network, Engineer]",https://cecinvestor.com/search/kitwe/feed/rss2/,"[Copperbelt, Energy, Corporation, PLC, CEC, in...",2014-10-10 14:32:52+00:00,aiciadmin,1,0,1,0,0,0,0,0,2,1
205,Copperbelt Energy,Careers,"[vacancy, Management, Accountant]",https://cecinvestor.com/search/kitwe/feed/rss2/,"[Copperbelt, Energy, Corporation, PLC, CEC, in...",2014-09-25 15:29:05+00:00,aiciadmin,1,0,1,0,0,0,0,0,2,1
206,Copperbelt Energy,Corporate Announcements,"[Sixteenth, AGM, presentation, CEC, Shareholders]",https://cecinvestor.com/search/kitwe/feed/rss2/,"[Directors, Copperbelt, Energy, Corporation, P...",2014-07-30 15:29:01+00:00,aiciadmin,1,0,1,0,0,0,0,0,2,1
207,Copperbelt Energy,Corporate Announcements,"[notice, Sixteenth, Annual, General, Meeting]",https://cecinvestor.com/search/kitwe/feed/rss2/,"[notice, give, Sixteenth, Annual, General, Mee...",2014-07-11 16:14:31+00:00,aiciadmin,1,0,1,0,0,0,0,0,2,1
208,Copperbelt Energy,News,"[CEC, management, commend, analyst]",https://cecinvestor.com/search/kitwe/feed/rss2/,"[CEC, commend, analyst, proactive, shareholder...",2014-02-26 12:44:54+00:00,aiciadmin,1,0,1,0,0,0,0,0,2,1
209,Copperbelt Energy,News,"[reminder, CEC, Investor, Open, Day]",https://cecinvestor.com/search/kitwe/feed/rss2/,"[Copperbelt, Energy, Corporation, PLC, CEC, in...",2014-02-06 07:37:34+00:00,aiciadmin,1,0,1,0,0,0,0,0,2,1


In [33]:
# Checking number of news with fake editiors
df_clean[df_clean['fake_author']==True].shape

(7302, 17)

In [34]:
# find instances where headline and description is not matching
df_clean[df_clean['dissimilar_head_desc']==True][:20]

,Source,Category,Headline,Link,Description,Date,Author,fake_author,dissimilar_head_desc,fake_url,click_bait,polar,excess_capitalization,susp_links,sens_description,fake_indicators,Target
18,Copperbelt Energy,Careers,"[CEC, Career, Opportunity, Instrumentation, Te...",https://cecinvestor.com/search/kitwe/feed/rss2/,"[Grade, Contract, Type, Permanent, location, K...",2024-05-27 15:05:31+00:00,Lovejoy Musundire,0,1,1,0,0,0,0,0,2,1
19,Copperbelt Energy,Careers,"[CEC, Career, Opportunity, Supply, Assistant]",https://cecinvestor.com/search/kitwe/feed/rss2/,"[Grade, Contract, Type, Permanent, location, K...",2024-05-27 15:04:41+00:00,Lovejoy Musundire,0,1,1,0,0,0,0,0,2,1
23,Copperbelt Energy,"Annual Report,Corporate Announcements,Download...","[CEC, Annual, Report, release]",https://cecinvestor.com/search/kitwe/feed/rss2/,"[pleased, great, progress, year, front, achiev...",2024-03-07 21:43:10+00:00,CEC Investor Relations,0,1,1,0,0,0,0,0,2,1
25,Copperbelt Energy,"Corporate Announcements,Downloads,Featured,AF","[CEC, release, Consolidated, audit, result, Fi...",https://cecinvestor.com/search/kitwe/feed/rss2/,"[Managing, Director, Owen, Silavwe, comment, t...",2024-03-01 12:34:38+00:00,Lovejoy Musundire,0,1,1,0,0,0,0,0,2,1
26,Copperbelt Energy,Careers,"[CEC, Career, Opportunity, technician, electri...",https://cecinvestor.com/search/kitwe/feed/rss2/,"[invite, application, suitably, qualified, inn...",2024-01-24 15:37:07+00:00,Lovejoy Musundire,0,1,1,0,0,0,0,0,2,1
27,Copperbelt Energy,Careers,"[CEC, Career, Opportunity, Manager, Digital, I...",https://cecinvestor.com/search/kitwe/feed/rss2/,"[Grade, Contract, Type, Permanent, location, K...",2023-11-30 06:04:12+00:00,Lovejoy Musundire,0,1,1,0,0,0,0,0,2,1
28,Copperbelt Energy,Careers,"[CEC, Career, Opportunity, engineer, Informati...",https://cecinvestor.com/search/kitwe/feed/rss2/,"[Grade, Contract, Type, Permanent, location, K...",2023-11-30 06:03:07+00:00,Lovejoy Musundire,0,1,1,0,0,0,0,0,2,1
29,Copperbelt Energy,Careers,"[CEC, Career, Opportunity, Advisor, Talent, Ma...",https://cecinvestor.com/search/kitwe/feed/rss2/,"[Grade, Contract, Type, Permanent, location, K...",2023-10-02 19:00:24+00:00,CEC Investor Relations,0,1,1,0,0,0,0,0,2,1
31,Copperbelt Energy,Careers,"[CEC, Career, Opportunity, Manager, HR, Operat...",https://cecinvestor.com/search/kitwe/feed/rss2/,"[Grade, Contract, Type, Permanent, location, K...",2023-10-02 18:43:57+00:00,CEC Investor Relations,0,1,1,0,0,0,0,0,2,1
32,Copperbelt Energy,Careers,"[CEC, Career, Opportunity, Creditors, Accountant]",https://cecinvestor.com/search/kitwe/feed/rss2/,"[invite, application, suitably, qualified, inn...",2023-09-06 06:56:20+00:00,Lovejoy Musundire,0,1,1,0,0,0,0,0,2,1


In [35]:
df_clean.head()

,Source,Category,Headline,Link,Description,Date,Author,fake_author,dissimilar_head_desc,fake_url,click_bait,polar,excess_capitalization,susp_links,sens_description,fake_indicators,Target
0,Copperbelt Energy,"Careers,Current Careers","[CEC, Career, Opportunity, GDP, Legal]",https://cecinvestor.com/search/kitwe/feed/rss2/,"[currently, career, opportunity, follow, field...",2024-11-20 16:31:18+00:00,Lovejoy Musundire,0,0,1,0,0,0,0,0,1,0
1,Copperbelt Energy,"Careers,Current Careers","[CEC, Career, Opportunity, Assurance, Speciali...",https://cecinvestor.com/search/kitwe/feed/rss2/,"[currently, career, opportunity, follow, field...",2024-11-20 15:15:28+00:00,Lovejoy Musundire,0,0,1,0,0,0,0,0,1,0
2,Copperbelt Energy,"Careers,Current Careers","[CEC, Career, Opportunity, Internal, Auditor, ...",https://cecinvestor.com/search/kitwe/feed/rss2/,"[currently, career, opportunity, follow, field...",2024-11-20 15:14:39+00:00,Lovejoy Musundire,0,0,1,0,0,0,0,0,1,0
3,Copperbelt Energy,"Careers,Current Careers","[CEC, Career, Opportunity, Engineer, protection]",https://cecinvestor.com/search/kitwe/feed/rss2/,"[currently, career, opportunity, follow, field...",2024-11-20 15:13:41+00:00,Lovejoy Musundire,0,0,1,0,0,0,0,0,1,0
4,Copperbelt Energy,"Careers,Current Careers","[CEC, Career, Opportunity, Mechanic, II]",https://cecinvestor.com/search/kitwe/feed/rss2/,"[currently, career, opportunity, follow, field...",2024-11-20 15:13:05+00:00,Lovejoy Musundire,0,0,1,0,0,0,0,0,1,0


### Categorization of the data
Most of the colletected data does not have a proper category and even for the ones with the category, they have been assigned wrong. Sometimes the category is filled with information from the description or ambiguous values.

Let's build a strategy for the categorization of each text sample.

In [39]:
class TextCategorizer:
    """
    A class to assign a set of pre-selected categories and 
    assign KNN based classifier to assign the nearest category using
    tf-idf vectorization
    """
    def __init__(self, data, n_neighbors=5, max_features=5000):
        self.data = data # pandas data frame
        self.n_neighbors = n_neighbors # no of nearest neighbours to consider
        self.max_features = max_features # maximum number of features to consider
        self.vectorizer = TfidfVectorizer(max_features=max_features) #tf-dif vectorizer
        self.knn = KNeighborsClassifier(n_neighbors=n_neighbors) # KNN classifier
        
        # Define category keywords directly in the class
        self.categories_keywords = {
            'sports': ['football', 'soccer', 'basketball', 'tennis', 'cricket', 'olympics', 'athlete', 'sports'],
            'politics': ['government', 'election', 'politician', 'policy', 'parliament', 'minister', 'president', 'vote'],
            'education': ['school', 'university', 'education', 'college', 'students', 'learning', 'teacher', 'scholarship'],
            'health and wellness': ['health', 'hospital', 'doctor', 'wellness', 'mental health', 'fitness', 'medicine', 'disease'],
            'development': ['development', 'infrastructure', 'construction', 'road', 'bridge', 'building', 'urbanization'],
            'narcotics': ['narcotics', 'drug', 'cocaine', 'heroin', 'meth', 'drug trafficking', 'illegal drugs'],
            'fashion': ['fashion', 'clothing', 'designer', 'runway', 'model', 'style', 'apparel', 'trends'],
            'career': ['job', 'career', 'employment', 'opportunity', 'work', 'recruitment', 'hiring', 'position'],
            'local news': ['local', 'community', 'city', 'town', 'village', 'municipality', 'neighborhood', 'region'],
            'economy news': ['economy', 'economic', 'finance', 'market', 'stocks', 'currency', 'inflation', 'gdp'],
            'business news': ['business', 'company', 'corporation', 'entrepreneur', 'startup', 'industry', 'investment', 'profit']
        }
        
    def prioritize_category(self, description):
        """Assign a single category based on highest keyword count."""

        description = " ".join(description)
        keyword_count = {}
        for category, keywords in self.categories_keywords.items():
            count = sum(description.lower().count(keyword) for keyword in keywords)
            if count > 0:
                keyword_count[category] = count
        return max(keyword_count, key=keyword_count.get) if keyword_count else 'uncategorized'
    
    def assign_single_categories(self):
        """Apply single category based on keyword prioritization."""
        self.data['Single_Category'] = self.data['Description'].apply(self.prioritize_category)

    def train_knn_classifier(self):
        """Train the KNN model to predict categories for uncategorized entries."""
        desc = self.data['Description'].apply(lambda x: " ".join(x))
        
        cat = self.data['Single_Category'] != 'uncategorized'
        uncat = self.data['Single_Category'] == 'uncategorized'
        
        # Train data
        X_train = desc[cat]
        y_train = self.data['Single_Category'][cat]

        # test data
        X_test = desc[uncat]
        
        # Convert text to TF-IDF vectors
        X_train_tfidf = self.vectorizer.fit_transform(X_train)
        
        # Train KNN classifier
        self.knn.fit(X_train_tfidf, y_train)
        
        # Predict uncategorized entries
        if any(uncat):
            X_test_tfidf = self.vectorizer.transform(X_test)
            y_pred = self.knn.predict(X_test_tfidf)
            self.data.loc[uncat, 'Single_Category'] = y_pred
    
    def categorize(self):
        """Run all categorization steps in sequence, and replace 'Category' with 'Single_Category'."""
        self.assign_single_categories()
        self.train_knn_classifier()
        
        return self.data

In [40]:
tob = TextCategorizer(df_clean)
df_cat = tob.categorize()

In [41]:
df_cat.head()

,Source,Category,Headline,Link,Description,Date,Author,fake_author,dissimilar_head_desc,fake_url,click_bait,polar,excess_capitalization,susp_links,sens_description,fake_indicators,Target,Single_Category
0,Copperbelt Energy,"Careers,Current Careers","[CEC, Career, Opportunity, GDP, Legal]",https://cecinvestor.com/search/kitwe/feed/rss2/,"[currently, career, opportunity, follow, field...",2024-11-20 16:31:18+00:00,Lovejoy Musundire,0,0,1,0,0,0,0,0,1,0,career
1,Copperbelt Energy,"Careers,Current Careers","[CEC, Career, Opportunity, Assurance, Speciali...",https://cecinvestor.com/search/kitwe/feed/rss2/,"[currently, career, opportunity, follow, field...",2024-11-20 15:15:28+00:00,Lovejoy Musundire,0,0,1,0,0,0,0,0,1,0,career
2,Copperbelt Energy,"Careers,Current Careers","[CEC, Career, Opportunity, Internal, Auditor, ...",https://cecinvestor.com/search/kitwe/feed/rss2/,"[currently, career, opportunity, follow, field...",2024-11-20 15:14:39+00:00,Lovejoy Musundire,0,0,1,0,0,0,0,0,1,0,career
3,Copperbelt Energy,"Careers,Current Careers","[CEC, Career, Opportunity, Engineer, protection]",https://cecinvestor.com/search/kitwe/feed/rss2/,"[currently, career, opportunity, follow, field...",2024-11-20 15:13:41+00:00,Lovejoy Musundire,0,0,1,0,0,0,0,0,1,0,career
4,Copperbelt Energy,"Careers,Current Careers","[CEC, Career, Opportunity, Mechanic, II]",https://cecinvestor.com/search/kitwe/feed/rss2/,"[currently, career, opportunity, follow, field...",2024-11-20 15:13:05+00:00,Lovejoy Musundire,0,0,1,0,0,0,0,0,1,0,career


In [42]:
# Now drop the unnecessary columns and keep only the relevant ones
df_fin = df_cat.drop(columns=["Category", "fake_author", "dissimilar_head_desc", "fake_url", "click_bait", "polar", "excess_capitalization", "susp_links",
                     "sens_description", "fake_indicators"])

In [43]:
df_fin.head()

,Source,Headline,Link,Description,Date,Author,Target,Single_Category
0,Copperbelt Energy,"[CEC, Career, Opportunity, GDP, Legal]",https://cecinvestor.com/search/kitwe/feed/rss2/,"[currently, career, opportunity, follow, field...",2024-11-20 16:31:18+00:00,Lovejoy Musundire,0,career
1,Copperbelt Energy,"[CEC, Career, Opportunity, Assurance, Speciali...",https://cecinvestor.com/search/kitwe/feed/rss2/,"[currently, career, opportunity, follow, field...",2024-11-20 15:15:28+00:00,Lovejoy Musundire,0,career
2,Copperbelt Energy,"[CEC, Career, Opportunity, Internal, Auditor, ...",https://cecinvestor.com/search/kitwe/feed/rss2/,"[currently, career, opportunity, follow, field...",2024-11-20 15:14:39+00:00,Lovejoy Musundire,0,career
3,Copperbelt Energy,"[CEC, Career, Opportunity, Engineer, protection]",https://cecinvestor.com/search/kitwe/feed/rss2/,"[currently, career, opportunity, follow, field...",2024-11-20 15:13:41+00:00,Lovejoy Musundire,0,career
4,Copperbelt Energy,"[CEC, Career, Opportunity, Mechanic, II]",https://cecinvestor.com/search/kitwe/feed/rss2/,"[currently, career, opportunity, follow, field...",2024-11-20 15:13:05+00:00,Lovejoy Musundire,0,career


In [44]:
df_fin.rename(columns={"Single_Category":"Category"}, inplace=True)

In [45]:
# reordering the column names:
df_fin = df_fin[["Source", "Category", "Headline", "Description", "Link", "Date", "Author", "Target"]]

# print the head
df_fin.head()

,Source,Category,Headline,Description,Link,Date,Author,Target
0,Copperbelt Energy,career,"[CEC, Career, Opportunity, GDP, Legal]","[currently, career, opportunity, follow, field...",https://cecinvestor.com/search/kitwe/feed/rss2/,2024-11-20 16:31:18+00:00,Lovejoy Musundire,0
1,Copperbelt Energy,career,"[CEC, Career, Opportunity, Assurance, Speciali...","[currently, career, opportunity, follow, field...",https://cecinvestor.com/search/kitwe/feed/rss2/,2024-11-20 15:15:28+00:00,Lovejoy Musundire,0
2,Copperbelt Energy,career,"[CEC, Career, Opportunity, Internal, Auditor, ...","[currently, career, opportunity, follow, field...",https://cecinvestor.com/search/kitwe/feed/rss2/,2024-11-20 15:14:39+00:00,Lovejoy Musundire,0
3,Copperbelt Energy,career,"[CEC, Career, Opportunity, Engineer, protection]","[currently, career, opportunity, follow, field...",https://cecinvestor.com/search/kitwe/feed/rss2/,2024-11-20 15:13:41+00:00,Lovejoy Musundire,0
4,Copperbelt Energy,career,"[CEC, Career, Opportunity, Mechanic, II]","[currently, career, opportunity, follow, field...",https://cecinvestor.com/search/kitwe/feed/rss2/,2024-11-20 15:13:05+00:00,Lovejoy Musundire,0


In [132]:
# Finally save the cleaned, annotated and categorized data
df_fin.to_csv("../../data/data-final-cleaned.csv", index=False)

In [46]:
test = df_fin['Headline'].iloc[0]
print(test)
print(" ".join(test))

['CEC', 'Career', 'Opportunity', 'GDP', 'Legal']
CEC Career Opportunity GDP Legal


In [47]:
# Also saving the data in pickle format to a preserve the data type structure
df_fin.to_pickle("../../data/data-final-cleaned.pkl")